In [1]:
# [STEP 1] Environment Setup & Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Define clean relative paths to avoid exposing local personal absolute paths
RAW_DATA_PATH = '../data/raw/hotel_bookings.csv' 
PROCESSED_DIR = '../data/processed/'              

# Create the processed directory path if it does not exist
if not os.path.exists(PROCESSED_DIR):
    os.makedirs(PROCESSED_DIR)

# Load the raw data and create a working copy to preserve the baseline dataset
if os.path.exists(RAW_DATA_PATH):
    df_raw = pd.read_csv(RAW_DATA_PATH)
    df = df_raw.copy()
    print(f"Dataset loaded. Initial Shape: {df.shape}")
else:
    print("Error: Please place 'hotel_bookings.csv' in the '../data/raw/' folder.")

Dataset loaded. Initial Shape: (119390, 32)


In [2]:
# [STEP 2] Data Cleaning & Feature Extraction
# Sanitize columns by removing leading/trailing whitespaces
df.columns = df.columns.str.strip() 

# 1. Agent Feature Parsing: Convert raw nominal ID into a binary flag and drop the original column
if 'agent' in df.columns:
    df['has_agent'] = df['agent'].notnull().astype(int)
    df.drop('agent', axis=1, inplace=True)

# 2. Company Feature Parsing: Convert into a binary flag safely by checking column existence
if 'company' in df.columns:
    df['has_company'] = df['company'].notnull().astype(int)
    df.drop('company', axis=1, inplace=True) 
else:
    if 'has_company' not in df.columns:
        df['has_company'] = 0

# 3. Strategic Imputation: Handle remaining missing attributes
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna('Unknown')

# 4. Advanced Filtering: Purge invalid "Ghost Booking" records where total guest count equals zero
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df = df[df['total_guests'] > 0]

print(f"Cleaning complete. Current records: {len(df)}")

Cleaning complete. Current records: 119210


In [3]:
# [STEP 3] Feature Engineering & Data Leakage Prevention
# 1. Feature Derivation: Construct customized continuous variables based on EDA insights
df['total_stay'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['is_family'] = ((df['children'] > 0) | (df['babies'] > 0)).astype(int)

# 2. Dimensionality Management: Group sparse country categories into 'Other' to prevent feature explosion
top_10_countries = df['country'].value_counts().nlargest(10).index
df['country'] = df['country'].apply(lambda x: x if x in top_10_countries else 'Other')

# 3. Target Leakage Defense: Explicitly drop columns determined post-booking outcome
leak_cols = ['reservation_status', 'reservation_status_date']
df.drop([c for c in leak_cols if c in df.columns], axis=1, inplace=True)

print("Feature engineering and leakage prevention applied.")

Feature engineering and leakage prevention applied.


In [4]:
# [STEP 4] One-Hot Encoding & Final Verification
# List all relevant categorical columns as per lead feedback
cat_cols = [
    'hotel', 'meal', 'market_segment', 'distribution_channel', 
    'deposit_type', 'customer_type', 'arrival_date_month', 'country',
    'reserved_room_type', 'assigned_room_type'
]

# Convert categorical features into numeric dummies
df_final = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Convert boolean dummies to 0/1 integers for model compatibility
bool_cols = df_final.select_dtypes(include=['bool']).columns
df_final[bool_cols] = df_final[bool_cols].astype(int)

# Final Verification: Ensure no 'object' types remain for modeling
remaining_objects = df_final.select_dtypes(include=['object']).columns
if remaining_objects.empty:
    print(f"Final dataset prepared. All features are numeric. Total columns: {df_final.shape[1]}")
else:
    print(f"Warning: Objects remaining -> {list(remaining_objects)}")

Final dataset prepared. All features are numeric. Total columns: 83


In [5]:
# [STEP 5] Export Processed Data with Clean Relative Paths Output
# 1. Classification Subset Generation: Export full digitized matrix for cancellation prediction
clf_output = os.path.join(PROCESSED_DIR, 'hotel_bookings_clf.csv')
df_final.to_csv(clf_output, index=False)
print("Saved classification dataset to `../data/processed/hotel_bookings_clf.csv`")

# 2. Regression Subset Generation: Filter out canceled records for revenue (ADR) estimation
# Enforce strict outlier bounding (0 < ADR < 500) for prediction stability
df_reg = df_final[df_final['is_canceled'] == 0].copy()
df_reg = df_reg[(df_reg['adr'] > 0) & (df_reg['adr'] < 500)]

reg_output = os.path.join(PROCESSED_DIR, 'hotel_bookings_reg.csv')
df_reg.to_csv(reg_output, index=False)
print("Saved regression dataset to `../data/processed/hotel_bookings_reg.csv`")

Saved classification dataset to `../data/processed/hotel_bookings_clf.csv`


Saved regression dataset to `../data/processed/hotel_bookings_reg.csv`


In [6]:
# [STEP 6] Core Preprocessing Pipeline Verification via src/processing.py
# Append the parent directory to the runtime path to recognize custom package structures
sys.path.append(os.path.abspath(os.path.join('..')))
from src.processing import preprocess_hotel_booking_data

# Execute the centralized automation module to audit and guarantee identical data results
preprocess_hotel_booking_data(raw_data_path=RAW_DATA_PATH, output_dir=PROCESSED_DIR)

STARTING HOTEL BOOKING DATA PREPROCESSING PIPELINE


Loaded raw dataset successfully. Initial Shape: (119390, 32)
Filtered zero-guest booking anomalies. Remaining records: 119210


Saved classification dataset to `../data/processed/hotel_bookings_clf.csv`


Saved regression dataset to `../data/processed/hotel_bookings_reg.csv`

FINAL PIPELINE VERIFICATION AUDIT
Raw dataset loaded successfully  : True (Shape: (119390, 32))
Classification dataset (CLF) shape: 119210 rows / 83 columns
Regression dataset (REG) shape    : 73386 rows / 83 columns
Total Remaining Missing Values   : 0
Total Remaining Object Columns   : 0
